# 04 · GenAI Tools Demo

* **Workshop:** AI for Actuaries — From Foundations to AI Agents
* **Session / Part:** S2.P2
* **Slides covered:** S2.P2.4, S2.P2.6, S2.P2.8
* **Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Sai Krishna Vadali
* **Workshop date:** 15 May 2026 · Hilton near Airport, Mumbai

## What this notebook does

Three reproducible Gemini API prompts that mirror Demos 1, 2, and 3 from the slide deck.

Demo 4 (Cursor IDE) is a live screen-share, not a notebook cell.

## Prerequisites

- Google account (for Colab)
- Gemini API key — set in **Colab Secrets** as `GEMINI_API_KEY` (free tier is sufficient)
- No local install required

## How to run

Top menu → **Runtime → Run all**. The first cell installs `google-genai`; the rest run without intervention.


## 0. Install

In [1]:
# Install google-genai. Pinned at 1.0+ — confirm against your Colab runtime
# at notebook freeze time.
!pip install -q google-genai

## 0.1 Standard imports and reproducibility

In [2]:
# === Standard imports ===
import json
import os
import datetime as dt
from pathlib import Path

# Reproducibility (LLM outputs are still non-deterministic by default —
# we set temperature=0 in calls below to get closer-to-stable results).
SEED = 42

# Notebook-specific imports below
# -------------------------------------------------------------
from google import genai
from google.genai import types

## 0.2 API setup

We read the Gemini API key from **Colab Secrets**. Add it in the left sidebar (key icon)
under the name `GOOGLE_API_KEY`. Local fallback: set the environment variable.

Pinning the model identifier matters — `gemini-2.5-flash` today is not the same model
in six months. Always pin in code.


In [3]:
# Read GOOGLE_API_KEY from Colab Secrets, fall back to env var.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    if "GOOGLE_API_KEY" not in os.environ:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. "
            "Add it via Colab Secrets (left sidebar key icon) "
            "or set the env variable."
        )

# Pin the model. Confirm at notebook-finalisation time and update if the
# stable identifier has changed.
MODEL_ID = "gemini-2.5-flash"

# Output directory.
OUT_DIR = Path("/content")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Single shared client.
client = genai.Client()

print(f"API key loaded. Model pinned to: {MODEL_ID}")


API key loaded. Model pinned to: gemini-2.5-flash


## 0.3 Prompt log

Per slide S2.P2.12: log every API call. The helper below appends a one-line
record to `/content/prompt_log.csv` for audit and reproducibility.


In [4]:
LOG_PATH = OUT_DIR / "prompt_log.csv"

def _log_call(demo_id: str, prompt: str, response_text: str) -> None:
    """Append a one-line record of the API call to the prompt log."""
    ts = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    header_needed = not LOG_PATH.exists()
    with LOG_PATH.open("a", encoding="utf-8") as f:
        if header_needed:
            f.write("timestamp_utc,demo_id,model,prompt_chars,response_chars\n")
        f.write(
            f"{ts},{demo_id},{MODEL_ID},"
            f"{len(prompt)},{len(response_text)}\n"
        )

print(f"Logger ready. Records will be appended to: {LOG_PATH}")


Logger ready. Records will be appended to: /content/prompt_log.csv


## 1. Demo 1 — risk-factor table for ABC Motor (GI)

**Slide:** S2.P2.4
**Goal:** Generate ten rating factors for an ABC Motor private car comprehensive
product, each tagged with directional impact and a one-line justification.
**Output format:** JSON

> ABC Insurer is hypothetical — see `00_personas_and_datasets.md`. Numbers are
> illustrative.


In [5]:
import json
from google import genai

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Prompt
prompt = """
Generate 10 rating factors for an Indian private car insurance product.

For each factor return:
- factor
- direction ("increase" or "decrease")
- justification

Return ONLY valid JSON.
"""

# Generate structured output
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
    },
)

# Parse JSON
factors = json.loads(response.text)

# Print formatted output
print(json.dumps(factors, indent=2))

[
  {
    "factor": "Vehicle Age",
    "direction": "decrease",
    "justification": "As the vehicle ages, its Insured Declared Value (IDV) decreases, leading to lower premiums for own-damage coverage."
  },
  {
    "factor": "Geographic Zone",
    "direction": "increase",
    "justification": "Vehicles registered in Tier-1 cities (Zone A) are subject to higher premiums due to increased traffic density and higher theft risk."
  },
  {
    "factor": "Engine Capacity (CC)",
    "direction": "increase",
    "justification": "Higher engine capacity typically correlates with higher repair costs and increased speed capabilities, warranting higher premiums."
  },
  {
    "factor": "No Claim Bonus (NCB)",
    "direction": "decrease",
    "justification": "A higher NCB percentage reflects a history of claim-free years, resulting in a direct discount on the premium."
  },
  {
    "factor": "Voluntary Deductible",
    "direction": "decrease",
    "justification": "Choosing a higher voluntary dedu

## 2. Demo 2 — UW guidelines for ABC Term Life (Life)

**Slide:** S2.P2.6
**Goal:** Draft underwriting guidelines for an ABC Term Life product covering
issue ages 25–55 and sums assured up to ₹1 crore (Indian retail market).
**Output format:** Markdown, 300–400 words

> Output is a draft. Numerical thresholds (NoMed/TeleMed/FullMed bands etc.) must
> be matched against ABC Life's live underwriting matrix before publication.


In [10]:
from google import genai

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Prompt
prompt = """
Draft underwriting guidelines for an Indian term life insurance product.

Include:
1. Eligibility
2. Medical underwriting
3. Financial underwriting
4. Smoker rules
5. PED handling

Assume:
- Entry age: 25–55
- Sum assured: up to ₹1 crore

Use simple markdown format.
"""

# Generate response
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

# Display nicely formatted markdown in Google Colab
display(Markdown(response.text))

These underwriting guidelines are designed for a standard term life insurance product in the Indian market with a maximum Sum Assured (SA) of ₹1 crore and an entry age bracket of 25–55.

---

### 1. Eligibility
*   **Age at Entry:** 25 to 55 years (as per last birthday).
*   **Maximum Maturity Age:** 75 years.
*   **Residency:** Must be a resident Indian (or NRI/OCI subject to specific geographical risk grading).
*   **Minimum SA:** ₹25 Lakhs.
*   **Maximum SA:** ₹1 Crore.
*   **Employment:** Salaried or Self-Employed with a valid ITR filing history.

---

### 2. Medical Underwriting
Underwriting is determined by the **Sum Assured** and the **Age** of the proposer.

*   **Standard Medical Requirements (Up to ₹1 Cr):**
    *   **Age 25–40:** Generally "Tele-Medical" (Tele-MER) or "Non-Medical" with a detailed Health Questionnaire (PHQ).
    *   **Age 41–55:** Compulsory Medical Examination (CME) including:
        *   Physical examination (Height, Weight, Blood Pressure).
        *   Routine Blood tests (CBC, HbA1c, Lipid Profile).
        *   Urine Routine Analysis.
        *   ECG (mandatory for 50+ or if BMI > 30).
*   **BMI (Body Mass Index):** Standard acceptance for BMI between 18.5 and 27.5. Ratings or loading apply for BMI > 30.

---

### 3. Financial Underwriting
To ensure the Sum Assured is commensurate with the proposer’s income (Human Life Value - HLV).

*   **Income Documentation:**
    *   **Salaried:** Last 3 months' salary slips and latest Form 16.
    *   **Self-Employed:** Last 2 years' ITR (with computation of income).
*   **Income Multiplier:**
    *   Maximum SA is capped at 10x to 15x of the Annual Income, depending on the age group.
*   **Non-Income Proof (NIP):** For SA up to ₹50 Lakhs, "Non-Medical" declaration may be accepted for applicants with verified stable employment, subject to internal credit scoring.

---

### 4. Smoker Rules
*   **Declaration:** Mandatory disclosure of tobacco/nicotine consumption (including cigarettes, bidis, gutka, and nicotine patches).
*   **Premium Loading:** 
    *   Tobacco users are classified as "Substandard" and charged higher premium rates (Tobacco/Smoker rates).
    *   **Cotinine Test:** If a proposer declares as a "Non-Smoker" but is flagged during medical screening (via urine cotinine test), they will be automatically reclassified as a "Smoker" with retroactive premium adjustments.

---

### 5. Pre-Existing Disease (PED) Handling
PEDs are evaluated based on the date of diagnosis, current control/treatment, and stability.

*   **Disclosure:** Proposer must disclose all past surgeries, chronic illnesses (Diabetes, Hypertension, Thyroid), and hospitalizations in the last 5 years.
*   **Treatment Protocols:**
    *   **Minor PEDs (e.g., Controlled Hypertension):** Standard rates or minimal loading.
    *   **Chronic PEDs (e.g., Diabetes):** Must be stable for at least 12–24 months. HbA1c reports required.
    *   **Major PEDs (e.g., Heart Disease, Cancer History):** Usually results in a "Decline" or "Postponement" of the application.
*   **Exclusions:** PEDs disclosed at the time of proposal may be subject to a "Waiting Period" or an "Extra Premium" (loading) depending on the underwriting assessment.

---

### General Terms
*   **Right to Decline:** The insurer reserves the right to decline any application based on adverse medical history, hazardous occupations (e.g., mining, high-altitude work), or non-verifiable financial documents.
*   **Incontestability:** As per Section 45 of the Insurance Act 1938, no policy can be called into question after 3 years on the grounds of misstatement, unless such misstatement was made fraudulently.

## 3. Demo 3 — ABC Health board summary (Health)

**Slide:** S2.P2.8
**Goal:** Take a JSON dict of model results from a hospitalisation-frequency
model and produce a 200-word executive summary suitable for the ABC Health
board pack.
**Output format:** Markdown, ~200 words, saved to `/content/demo_3_output.md`.

The pattern: structured numbers in, structured prose out. No statistic in the
output that wasn't in the input.


In [9]:
import json
from google import genai
from IPython.display import Markdown, display

# Create Gemini client
client = genai.Client()

# Select model
MODEL_ID = "gemini-3.1-flash-lite"

# Example health insurance model results
model_results = {
    "model_name": "Hospitalisation Risk Model",
    "members_modelled": 120000,
    "model_accuracy_auc": 0.81,
    "high_risk_member_lift": 3.2,
    "top_risk_drivers": [
        "Member Age",
        "Diabetes History",
        "Previous Hospitalisation"
    ],
    "fairness_gap_percent": 1.8
}

# Prompt
prompt = f"""
Write an executive summary for the board of a health insurance company.

Use the following model results:
{json.dumps(model_results, indent=2)}

Requirements:
- Around 150 words
- Use simple business language
- Explain the model performance clearly
- Mention the top risk drivers
- Mention the fairness gap
- End with a recommendation
- Return markdown only
"""

# Generate response
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

# Display nicely formatted markdown in Google Colab
display(Markdown(response.text))

### Executive Summary: Hospitalisation Risk Model Implementation

We have successfully developed and validated the "Hospitalisation Risk Model" to proactively manage our member health outcomes. Applied across a cohort of 120,000 members, the model demonstrates strong predictive performance with an AUC of 0.81. Notably, the model achieves a 3.2x lift in identifying high-risk members, allowing our care management teams to intervene significantly earlier than previous methods.

Our analysis identifies three primary drivers of hospitalisation risk: **Member Age**, **Diabetes History**, and **Previous Hospitalisation**. Furthermore, the model meets our internal compliance standards, maintaining a minimal fairness gap of just 1.8%, ensuring equitable risk assessment across all demographic groups.

Given these results, we recommend immediate operational deployment of this model to support our preventative care programs. Integrating these insights will enable more efficient resource allocation, reduce avoidable hospital admissions, and ultimately improve the long-term health trajectory of our members while lowering overall claims costs.

## Wrap-up

You should now be able to:

- Call the Gemini API from Colab with a pinned model identifier and an API key loaded from Colab Secrets.
- Run a structured-output prompt that returns parseable JSON (Demo 1).
- Run a freeform-markdown prompt with explicit structure, length, and voice constraints (Demo 2).
- Feed a JSON of model results into a prompt and produce a board-ready prose summary that uses only the numbers you fed in (Demo 3).


**Where to next:** open `05_agents_intro.ipynb` for the minimal Agno + Gemini hello-agent that powers Part 3 of Session 2.

**Companion slides:** S2.P2.4 (Demo 1), S2.P2.6 (Demo 2), S2.P2.8 (Demo 3).

**Validation reminder:** The actuary owns the number, not the model. Before any output here goes near a regulatory deliverable, verify every figure against a real source.
